In [ ]:
import mitsuba as mi

mi.set_variant("cuda_acoustic", 'llvm_ad_acoustic')

print(f'Using variant: {mi.variant()}')

# sogar mit Info:
mi.set_log_level(mi.LogLevel.Info)

In [ ]:
renderer = mi.load_dict({
    "type": "acoustic_path",
    "max_depth": -1,
    "max_time": 0.2,
    "max_energy_loss": 60,
    "acoustic_medium":
        {
            "temperature": 20,
            "relative_humidity": 0.7,
            "atmospheric_pressure": 80000,
            #"saturation_vapor_pressure": ATMO_SATURATION_VAPOR_PRESSURE,
            "co2_ppm": 450,
            "speed_of_sound_method": 'auto',
            "apply_attenuation": True,
        }
})

In [ ]:
from pathlib import Path

path = Path("./l_room_1000m3_sealed.obj")

# ------------------------------------------------------------------
# L-room dimensions
# ------------------------------------------------------------------

a = 5.0
b = 10.0
c = 10.0
d = 5.0
height = 10.0

# L-shaped footprint, counter-clockwise when viewed from above (+Z up).
#
#       6 (0,b) -------- 5 (a+c,b)
#         |                |
#         |                |
#         |      3 (a,b-d)-4 (a+c,b-d)
#         |         |
#         |         |
#       1 (0,0) --- 2 (a,0)
#
# This is the ACTUAL 1-indexed order the vertex list below produces.
# (Any triangulation below must reference THIS numbering, not a
# re-derived/re-labeled one -- that mismatch was the original bug.)

footprint = [
    (0.0, 0.0),      # 1
    (a, 0.0),        # 2
    (a, b - d),      # 3  <- the reflex (concave) vertex
    (a + c, b - d),  # 4
    (a + c, b),       # 5
    (0.0, b),         # 6
]

# ------------------------------------------------------------------
# Create vertices
# ------------------------------------------------------------------

verts = []

# Bottom vertices (indices 1..6)
for x, y in footprint:
    verts.append((x, y, 0.0))

# Top vertices (indices 7..12)
for x, y in footprint:
    verts.append((x, y, height))

# Coordinate convention: OBJ Y-up. Map (x, y, z) -> (x, z, -y).
# This transform has determinant +1, so it preserves winding/handedness --
# normals computed as "outward" in (x, y, z) space stay outward after this.
verts = [(x, z, y-10) for x, y, z in verts]

n = len(footprint)

# ------------------------------------------------------------------
# Create faces
# ------------------------------------------------------------------

faces = []

# --------------------------------------------------------------
# Walls (unchanged -- already produce correct outward normals)
# --------------------------------------------------------------

for i in range(n):
    j = (i + 1) % n

    bottom_i = i + 1
    bottom_j = j + 1
    top_i = i + 1 + n
    top_j = j + 1 + n

    faces.append((bottom_i, bottom_j, top_j, top_i))

# --------------------------------------------------------------
# Floor + Ceiling
# --------------------------------------------------------------
#
# The footprint has exactly one reflex vertex: 3 = (a, b-d).
# Splitting along diagonal 3-6 gives two convex quads:
#   Quad A: 1, 2, 3, 6
#   Quad B: 3, 4, 5, 6
# Each quad triangulates into exactly 2 triangles -> 4 total
# (a simple hexagon always triangulates into n-2 = 4 triangles,
# never 5).
#
# In (x, y) space with this vertex order, winding (1,2,3,4,5,6)
# is CCW -> normal +Z (up). So:
#   - Ceiling (top, z=height) wants normal +Z -> keep this winding.
#   - Floor (bottom, z=0) wants normal -Z (outward/down) -> reverse it.

floor_faces = [
    (1, 3, 2),
    (1, 6, 3),
    (3, 5, 4),
    (3, 6, 5),
]

ceiling_faces = [(v1 + n, v2 + n, v3 + n) for v1, v2, v3 in
                 [(1, 2, 3), (1, 3, 6), (3, 4, 5), (3, 5, 6)]]

faces.extend(floor_faces)
faces.extend(ceiling_faces)

# ------------------------------------------------------------------
# Write OBJ
# ------------------------------------------------------------------

obj_lines = [
    "# Sealed L-room",
    "# Dimensions:",
    f"# a = {a} m",
    f"# b = {b} m",
    f"# c = {c} m",
    f"# d = {d} m",
    f"# height = {height} m",
    "# Volume = 1000 m^3",
    "# Closed mesh: 6 walls + floor (4 tris) + ceiling (4 tris)",
    "# No source or microphone",
    "",
    "o l_room_1000m3_sealed",
]

for v in verts:
    obj_lines.append(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}")

for f in faces:
    obj_lines.append("f " + " ".join(map(str, f)))

path.write_text("\n".join(obj_lines) + "\n")

print(f"Written to: {path}")
print(f"Vertices: {len(verts)}")
print(f"Faces: {len(faces)}  (6 wall quads + 4 floor tris + 4 ceiling tris)")